# CS231 – Phân loại độ chín của chuối bằng **HOG (tính theo từng kênh màu RGB)**

**Phụ trách:** thử nghiệm HOG · **Dữ liệu vào:** bộ đã tiền xử lý `Banana_Merged_Preprocessed` (thư mục `dataset/{train,val,test}/{lớp}/` + `reports/final_dataset_split_mapping.csv`, ảnh 224×224) · **Tập chính:** BananaID · **Kiểm định ngoài:** BananaImageBD (và BananaRipeness_GitHub nếu có)

**Cách hiểu "HOG: chia bin theo màu RGB" trong notebook này:** histogram hướng gradient (các *bin hướng*) được tính **riêng cho từng kênh R, G, B** rồi nối lại (`hog_rgb_concat`). Để biết việc tách theo màu có thật sự đáng giá, notebook so với hai đối chứng:

| Biến thể | Mô tả |
|---|---|
| `hog_gray` | HOG trên ảnh xám (HOG chuẩn, đối chứng) |
| `hog_rgb_concat` | HOG riêng cho R, G, B rồi nối (đúng ý "chia bin theo màu RGB") |
| `hog_rgb_maxgrad` | HOG màu kiểu Dalal–Triggs: tại mỗi pixel lấy kênh có gradient lớn nhất |

Mỗi biến thể được thử với `pixels_per_cell` ∈ {16, 32} và 3 bộ phân loại (KNN, SVM tuyến tính, SVM RBF – tương ứng bài tập 4 của môn: so sánh KNN và bộ phân loại tuyến tính trên HOG).

**Giao thức đánh giá (3 tầng, không tầng nào dùng dữ liệu của tầng sau):**
1. **Chọn cấu hình:** gộp `train + val` thành tập *dev*, `StratifiedGroupKFold` 5 phần chia **theo quả**; siêu tham số (k, C, γ) chọn bằng `GridSearchCV` + `GroupKFold` lồng bên trong phần train của mỗi fold.
2. **Tập test giữ lại** (cùng miền BananaID): cấu hình tốt nhất của mỗi biến thể được huấn luyện lại trên toàn bộ dev rồi đánh giá **một lần**.
3. **Kiểm định ngoài** (BananaImageBD, BananaRipeness_GitHub): đánh giá **một lần**, không dùng để chọn gì.

**Lưu ý quan trọng về cách chia train/val/test của bước tiền xử lý:** bước đó gom nhóm theo dHash ảnh, nên **4 góc chụp của cùng một quả vẫn có thể rơi vào các split khác nhau** (các góc nhìn khác nhau thì dHash khác nhau). Notebook tự kiểm tra rò rỉ này (mục 1) và, nếu có, **chia lại theo quả** (`SPLIT_MODE='auto'`), đồng thời ghi số liệu rò rỉ ra log để cả nhóm biết.

**Độ đo:** accuracy, macro-precision, macro-recall, macro-F1, confusion matrix (+ accuracy lệch ≤ 1 mức vì độ chín là thang có thứ tự; + độ trễ).

**Nhật ký và thống kê** được lưu toàn bộ vào thư mục output (bảng cuối notebook liệt kê) để dùng khi viết nhận xét.

In [ ]:
import os, re, sys, json, time, hashlib, logging, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import sklearn, skimage
from skimage.feature import hog
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             classification_report, silhouette_score)
import joblib
try:
    from IPython.display import display
except Exception:
    display = print
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': .3})

# ============================== CẤU HÌNH ==============================
DATA_ROOT = os.environ.get('CS231_DATA_ROOT') or None  # thư mục chứa dataset/ và reports/ ; None = tự tìm trong /kaggle/input
TRAIN_DATASETS    = ['BananaID']                                      # dữ liệu huấn luyện/chọn mô hình/test cùng miền
EXTERNAL_DATASETS = ['BananaImageBD', 'BananaRipeness_GitHub']       # kiểm định ngoài (dataset nào không có thì bỏ qua)
SPLIT_MODE = 'auto'          # 'auto' = dùng split của bước tiền xử lý nếu không rò rỉ theo quả, ngược lại chia lại theo quả
                             # 'preprocessed' = luôn dùng split có sẵn ; 'regroup' = luôn chia lại theo quả
EXTERNAL_MAX_PER_CLASS = None                          # None = dùng hết; hoặc số ảnh tối đa/lớp/dataset

LABELS = ['Unripe', 'Half_Ripe', 'Ripe', 'Overripe']   # thứ tự chín tăng dần
SEED, N_OUTER, N_INNER = 231, 5, 3
IMG_SIZE = (128, 128)                                  # (rộng, cao) sau khi resize từ 224×224, dùng cho mọi ảnh
HOG_ORIENT, HOG_CELLS_PER_BLOCK = 9, (2, 2)
PPC_LIST = [16, 32]                                    # pixels_per_cell ([32] chạy nhanh hơn nhiều)
HOG_VARIANTS = ['hog_gray', 'hog_rgb_concat', 'hog_rgb_maxgrad']
CLASSIFIERS = ['KNN', 'SVM-linear', 'SVM-RBF']
LEARNING_CURVE_FRACS = [0.2, 0.4, 0.6, 0.8, 1.0]
N_JOBS = -1
# =====================================================================

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs_hog')
OUT_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger('hog')
logger.setLevel(logging.INFO); logger.handlers.clear()
for h in (logging.StreamHandler(sys.stdout), logging.FileHandler(OUT_DIR / 'run_log.txt', mode='w', encoding='utf-8')):
    h.setFormatter(logging.Formatter('%(asctime)s | %(message)s', '%H:%M:%S')); logger.addHandler(h)
def log(*a): logger.info(' '.join(str(x) for x in a))

CONFIG = dict(TRAIN_DATASETS=TRAIN_DATASETS, EXTERNAL_DATASETS=EXTERNAL_DATASETS, SPLIT_MODE=SPLIT_MODE, LABELS=LABELS,
              SEED=SEED, N_OUTER=N_OUTER, N_INNER=N_INNER, IMG_SIZE=IMG_SIZE, HOG_ORIENT=HOG_ORIENT,
              HOG_CELLS_PER_BLOCK=HOG_CELLS_PER_BLOCK, PPC_LIST=PPC_LIST, HOG_VARIANTS=HOG_VARIANTS,
              CLASSIFIERS=CLASSIFIERS, EXTERNAL_MAX_PER_CLASS=EXTERNAL_MAX_PER_CLASS)
RUN_META = dict(started=time.strftime('%Y-%m-%d %H:%M:%S'), python=sys.version.split()[0], platform=platform.platform(),
                numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__, skimage=skimage.__version__,
                opencv=cv2.__version__, config=CONFIG, timings={})
log('Output ->', OUT_DIR.resolve()); log('Phiên bản:', {k: RUN_META[k] for k in ('python', 'numpy', 'sklearn', 'skimage', 'opencv')})

## 1. Đọc dữ liệu đã tiền xử lý, kiểm tra rò rỉ, chia tập

- Cấu trúc đầu vào: `dataset/{train,val,test}/{Unripe,Half_Ripe,Ripe,Overripe}/{Dataset}_{md5}.png` và `reports/final_dataset_split_mapping.csv` (cột `dataset, path, label, md5, group, split, out_path`; `path` là đường dẫn **ảnh gốc** nên còn tên file `Giống_ĐộChín_Góc_ID`).
- Nhãn = tên thư mục lớp; **dataset** = cột `dataset` của file mapping (hoặc tiền tố tên file nếu không có mapping).
- **Nhóm chống rò rỉ = một quả chuối**: lấy từ tên file gốc `Giống_ĐộChín_Góc_ID` (chỉ BananaID có), rồi **hợp nhất** với nhóm dHash của bước tiền xử lý bằng union-find. Mọi CV/chia tập đều theo nhóm này.
- Kiểm tra rò rỉ của split có sẵn → nếu có thì chia lại (xem `SPLIT_MODE`); kết quả lưu ở `split_used.csv`, `split_leakage_check.csv`.
- Mọi ảnh được đọc **một lần**, resize về `IMG_SIZE` và giữ trong RAM. Thống kê lưu ở `dataset_manifest.csv`, `dataset_stats.csv`.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLITS = ['train', 'val', 'test']
NAME_RE = re.compile(r'(Ambon|Cavendish|Raja|Saba|Mas)[_\- ]+(?:Half[_\-]?Ripe|Overripe|Ripe|Unripe)'
                     r'[_\- ]+(Top|Bottom|Left|Right)[_\- ]+(\d+)', re.I)
base_name = lambda p: re.split(r'[\\/]', str(p))[-1]

def find_data():
    """-> (thư mục dataset/ chứa train|val|test, đường dẫn file mapping hoặc None)"""
    def ds_of(parent):
        for cand in (parent / 'dataset', parent):
            if all((cand / s).is_dir() for s in SPLITS): return cand
    bases = [Path(DATA_ROOT)] if DATA_ROOT else [Path('/kaggle/input'), Path('.'), Path('..')]
    for b in bases:
        if not b.exists(): continue
        for mp in sorted(b.rglob('final_dataset_split_mapping.csv'), key=lambda p: len(p.parts)):
            d = ds_of(mp.parent.parent)
            if d: return d, mp
    for b in bases:
        if not b.exists(): continue
        for p in sorted(b.rglob('train'), key=lambda p: len(p.parts)):
            if p.is_dir() and ds_of(p.parent): return ds_of(p.parent), None
    return None, None

def read_rgb(path):
    img = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_UNCHANGED)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 4:
        a = img[:, :, 3:4].astype(np.float32) / 255
        img = (img[:, :, :3] * a + 255 * (1 - a)).astype(np.uint8)
    return cv2.cvtColor(cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)

t0 = time.time()
DS_DIR, MAP_CSV = find_data()
assert DS_DIR is not None, ('Không tìm thấy thư mục dataset/{train,val,test}. Thêm dataset đã tiền xử lý vào notebook '
                            '(Add Input) hoặc đặt DATA_ROOT ở cell cấu hình.')
log('Thư mục ảnh:', DS_DIR, '| file mapping:', MAP_CSV)

if MAP_CSV is not None:
    mp = pd.read_csv(MAP_CSV)
    need = {'dataset', 'path', 'label', 'md5', 'group', 'split', 'out_path'}
    assert need <= set(mp.columns), f'File mapping thiếu cột {need - set(mp.columns)}'
    df = pd.DataFrame(dict(dataset=mp.dataset, label=mp.label, split=mp.split, md5=mp.md5, dh_group=mp.dataset + '#' + mp.group.astype(str),
                           orig_file=mp.path.map(base_name), out_file=mp.out_path.map(base_name),
                           orig_w=mp.get('w'), orig_h=mp.get('h')))
else:
    log('!! KHÔNG có file mapping -> không biết tên file gốc, không gom được 4 góc của một quả: nhóm = từng ảnh. '
        'CV/test của BananaID sẽ bị rò rỉ và cao giả. Hãy thêm reports/final_dataset_split_mapping.csv vào dataset.')
    rows = [dict(dataset=p.name.rsplit('_', 1)[0], label=p.parent.name, split=p.parent.parent.name, md5=p.stem.rsplit('_', 1)[-1],
                 dh_group=p.stem, orig_file=p.name, out_file=p.name, orig_w=np.nan, orig_h=np.nan)
            for s in SPLITS for p in sorted((DS_DIR / s).rglob('*')) if p.suffix.lower() in IMG_EXT]
    df = pd.DataFrame(rows)
df['path'] = [str(DS_DIR / s / l / f) for s, l, f in zip(df.split, df.label, df.out_file)]
exists = np.array([Path(p).exists() for p in df.path])
if (~exists).any(): log(f'!! {int((~exists).sum())} ảnh có trong mapping nhưng không có trên đĩa -> bỏ qua')
df = df[exists & df.label.isin(LABELS)].reset_index(drop=True)
log('Số ảnh theo dataset × split:\n' + pd.crosstab(df.dataset, df.split).reindex(columns=SPLITS).to_string())
unused = sorted(set(df.dataset) - set(TRAIN_DATASETS) - set(EXTERNAL_DATASETS))
if unused: log('Dataset có mặt nhưng không dùng trong notebook này:', unused)

m = df.orig_file.map(lambda f: NAME_RE.search(f))
df['variety'] = [x.group(1).title() if x else 'unknown' for x in m]
df['view'] = [x.group(2).title() if x else 'unknown' for x in m]
df['banana'] = [f'{x.group(1).title()}_{l}_{x.group(3)}' if x else None for x, l in zip(m, df.label)]
df['file'] = df.orig_file

# --- nhóm chống rò rỉ = hợp nhất nhóm dHash và quả chuối (union-find) ---
parent = {}
def find(a):
    parent.setdefault(a, a)
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
for dh, ba in zip(df.dh_group, df.banana):
    find(dh)
    if isinstance(ba, str): parent[find(dh)] = find('banana:' + ba)
roots = [find(dh) for dh in df.dh_group]
df['group'] = pd.factorize(pd.Series(roots))[0].astype(str)
mixed = df.groupby('group').label.nunique() > 1
if mixed.any():     # ảnh gần giống nhau (dHash) nhưng khác nhãn: giữ chung nhóm để không tách chúng giữa train/test; ghi lại để soát nhãn
    df[df.group.isin(mixed[mixed].index)][['dataset', 'file', 'label', 'group']].sort_values('group').to_csv(OUT_DIR / 'mixed_label_groups.csv', index=False)
    log(f'!! {int(mixed.sum())} nhóm chứa ảnh gần giống nhau nhưng KHÁC nhãn (xem mixed_label_groups.csv) – có thể nhiễu nhãn')
log('Nhóm lớn nhất:', int(df.group.value_counts().max()), 'ảnh | số nhóm:', df.group.nunique(), '/', len(df), 'ảnh')

# --- tập chính: kiểm tra rò rỉ của split có sẵn ---
pool = df[df.dataset.isin(TRAIN_DATASETS)].reset_index(drop=True)
assert len(pool), f'Không có ảnh của {TRAIN_DATASETS} trong dữ liệu'
leak_g = pool.groupby('group').split.nunique() > 1
leak_img = pool.group.map(leak_g)
leak_rows = []
for s in SPLITS:
    sub = pool[pool.split == s]
    others = set(pool[pool.split != s].group)
    leak_rows.append(dict(split=s, n_images=len(sub), n_groups=sub.group.nunique(),
                          images_with_group_in_other_split=int(sub.group.isin(others).sum()),
                          frac=float(sub.group.isin(others).mean())))
leak_tab = pd.DataFrame(leak_rows); leak_tab.to_csv(OUT_DIR / 'split_leakage_check.csv', index=False); display(leak_tab.round(3))
n_leak = int(leak_g.sum())
log(f'Rò rỉ trong split có sẵn của bước tiền xử lý: {n_leak}/{pool.group.nunique()} nhóm (quả) nằm ở >1 split; '
    f'ảnh test có "anh em" cùng quả trong train/val: {leak_tab.set_index("split").frac["test"]:.1%}')
use_regroup = SPLIT_MODE == 'regroup' or (SPLIT_MODE == 'auto' and n_leak > 0)
if use_regroup:
    ystrat = pool.label.values
    def fold_of(idx, n):
        sg = StratifiedGroupKFold(n, shuffle=True, random_state=SEED)
        tr, te = next(sg.split(np.zeros(len(idx)), ystrat[idx], pool.group.values[idx])); return idx[tr], idx[te]
    rest, te_i = fold_of(np.arange(len(pool)), 7)          # test ≈ 1/7, như bước tiền xử lý
    tr_i, va_i = fold_of(rest, 6)                           # val ≈ 1/6 của phần còn lại
    pool['split_used'] = 'train'; pool.loc[va_i, 'split_used'] = 'val'; pool.loc[te_i, 'split_used'] = 'test'
    log('-> ĐÃ CHIA LẠI train/val/test theo quả (SPLIT_MODE =', SPLIT_MODE + ')')
else:
    pool['split_used'] = pool.split
    log('-> Dùng nguyên split của bước tiền xử lý (SPLIT_MODE =', SPLIT_MODE + ')')
assert (pool.groupby('group').split_used.nunique() == 1).all() or not use_regroup, 'Rò rỉ sau khi chia lại!'
pool[['dataset', 'file', 'label', 'group', 'split', 'split_used']].to_csv(OUT_DIR / 'split_used.csv', index=False)

df_main = pool[pool.split_used.isin(['train', 'val'])].reset_index(drop=True)      # dev = train + val
df_test = pool[pool.split_used == 'test'].reset_index(drop=True)
df_ext = df[df.dataset.isin(EXTERNAL_DATASETS)].drop_duplicates('md5').reset_index(drop=True)
if EXTERNAL_MAX_PER_CLASS:
    df_ext = df_ext.groupby(['dataset', 'label'], group_keys=False).apply(
        lambda g: g.sample(min(len(g), EXTERNAL_MAX_PER_CLASS), random_state=SEED)).reset_index(drop=True)
bad = set(df_ext.md5) & set(pool.md5)
if bad: log(f'!! {len(bad)} ảnh ngoài trùng MD5 với tập chính -> loại khỏi tập ngoài'); df_ext = df_ext[~df_ext.md5.isin(bad)].reset_index(drop=True)

def load_all(d): return np.stack([read_rgb(p) for p in d.path])
IM_MAIN, IM_TEST = load_all(df_main), load_all(df_test)
IM_EXT = load_all(df_ext) if len(df_ext) else None
lab2i = {l: i for i, l in enumerate(LABELS)}
y, y_test = df_main.label.map(lab2i).values, df_test.label.map(lab2i).values
y_ext = df_ext.label.map(lab2i).values if len(df_ext) else None
groups = df_main.group.to_numpy(dtype=object)

pd.concat([df_main.assign(set='dev'), df_test.assign(set='test'), df_ext.assign(set='external')]).drop(columns=['out_file']).to_csv(
    OUT_DIR / 'dataset_manifest.csv', index=False)

# --- thống kê dữ liệu ---
stats = []
for name, d in (('dev', df_main), ('test', df_test), ('external', df_ext)):
    for ds in d.dataset.unique():
        for l in LABELS:
            g = d[(d.dataset == ds) & (d.label == l)]
            stats.append(dict(set=name, dataset=ds, label=l, n_images=len(g), n_groups=g.group.nunique(),
                              n_bananas=g.banana.nunique(), md5_dups=int(g.md5.duplicated().sum()),
                              orig_size=(f'{int(g.orig_w.mode().iloc[0])}x{int(g.orig_h.mode().iloc[0])}' if len(g) and g.orig_w.notna().any() else '')))
stats = pd.DataFrame(stats); stats.to_csv(OUT_DIR / 'dataset_stats.csv', index=False); display(stats)
cnt = lambda d: d.label.value_counts().reindex(LABELS).fillna(0).astype(int).to_dict()
log(f'DEV {len(df_main):,} ảnh / {df_main.banana.nunique()} quả {cnt(df_main)} | TEST {len(df_test):,} ảnh / {df_test.banana.nunique()} quả {cnt(df_test)}')
log('Ảnh/quả trong dev:', df_main.groupby('group').size().value_counts().sort_index().to_dict(),
    '| giống:', df_main.variety.value_counts().to_dict(), '| góc chụp:', df_main.view.value_counts().to_dict())
if len(df_ext):
    for ds, g in df_ext.groupby('dataset'): log(f'NGOÀI {ds}: {len(g):,} ảnh {cnt(g)}')
else:
    log('Không có ảnh ngoài -> bỏ qua kiểm định ngoài.')
if df_main.md5.duplicated().any(): log(f'!! {int(df_main.md5.duplicated().sum())} ảnh trùng MD5 trong tập dev')
RUN_META['timings']['load_s'] = round(time.time() - t0, 1)
RUN_META['data'] = dict(ds_dir=str(DS_DIR), mapping=str(MAP_CSV), regrouped=bool(use_regroup), leaking_groups_in_given_split=n_leak,
                        n_dev=len(df_main), n_test=len(df_test), n_ext=len(df_ext))

## 2. Trích đặc trưng HOG

Tham số: ảnh `IMG_SIZE`, 9 bin hướng (0–180°), block 2×2 cell, chuẩn hóa `L2-Hys`. Số chiều đặc trưng: `hog_gray` và `hog_rgb_maxgrad` cùng số chiều; `hog_rgb_concat` gấp 3 lần.

In [ ]:
def hog_feat(img, variant, ppc):
    kw = dict(orientations=HOG_ORIENT, pixels_per_cell=(ppc, ppc), cells_per_block=HOG_CELLS_PER_BLOCK, block_norm='L2-Hys')
    if variant == 'hog_gray':
        return hog(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), **kw)
    if variant == 'hog_rgb_concat':
        return np.concatenate([hog(img[:, :, c], **kw) for c in range(3)])
    if variant == 'hog_rgb_maxgrad':
        return hog(img, channel_axis=-1, **kw)
    raise ValueError(variant)

def hog_matrix(imgs, variant, ppc):
    t = time.perf_counter()
    X = np.stack([hog_feat(im, variant, ppc) for im in imgs]).astype(np.float32)
    return X, (time.perf_counter() - t) * 1000 / len(imgs)

KEYS = [(v, p) for v in HOG_VARIANTS for p in PPC_LIST]
FEAT, EXTRACT_MS = {}, {}
for v, p in KEYS:
    FEAT[(v, p)], EXTRACT_MS[(v, p)] = hog_matrix(IM_MAIN, v, p)
    log(f'{v:<16} ppc={p:<3} dims={FEAT[(v, p)].shape[1]:<6} trích đặc trưng {EXTRACT_MS[(v, p)]:.2f} ms/ảnh')
RUN_META['timings']['features_s'] = round(time.time() - t0 - RUN_META['timings']['load_s'], 1)
pd.DataFrame([dict(variant=v, ppc=p, dims=FEAT[(v, p)].shape[1], extract_ms_per_img=EXTRACT_MS[(v, p)],
                   mean=float(FEAT[(v, p)].mean()), std=float(FEAT[(v, p)].std()),
                   sparsity=float((FEAT[(v, p)] == 0).mean())) for v, p in KEYS]).to_csv(OUT_DIR / 'feature_info.csv', index=False)

## 3. Phân tích dữ liệu / đặc trưng (để giải thích kết quả về sau)

- **Ảnh minh hoạ HOG** (ảnh gốc, HOG xám, HOG riêng từng kênh R/G/B).
- **Thống kê gradient theo lớp và theo kênh màu** (độ lớn gradient trung bình, phân bố 9 bin hướng) – đây chính là "bin theo màu RGB": cho biết kênh nào mang thông tin biên/vết (chấm nâu, sọc) nhiều hơn ở từng mức chín.
- **Không gian đặc trưng**: PCA 2 chiều tô theo lớp và theo **giống chuối**; silhouette theo lớp so với theo giống (nếu đặc trưng gom theo giống nhiều hơn theo độ chín thì phân loại độ chín sẽ khó và dễ bị "đường tắt").

In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(LABELS), 5, figsize=(13, 2.7 * len(LABELS)))
for r, l in enumerate(LABELS):
    i = rng.choice(np.where(y == r)[0]); img = IM_MAIN[i]
    axes[r, 0].imshow(img); axes[r, 0].set_title(f'{l}\n{df_main.file[i]}', fontsize=7)
    _, hg = hog(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), orientations=HOG_ORIENT, pixels_per_cell=(16, 16),
                cells_per_block=HOG_CELLS_PER_BLOCK, visualize=True)
    axes[r, 1].imshow(hg, cmap='gray'); axes[r, 1].set_title('HOG xám', fontsize=8)
    for c, nm in enumerate('RGB'):
        _, hc = hog(img[:, :, c], orientations=HOG_ORIENT, pixels_per_cell=(16, 16), cells_per_block=HOG_CELLS_PER_BLOCK, visualize=True)
        axes[r, 2 + c].imshow(hc, cmap='gray'); axes[r, 2 + c].set_title(f'HOG kênh {nm}', fontsize=8)
for a in axes.ravel(): a.axis('off')
plt.tight_layout(); fig.savefig(OUT_DIR / 'hog_visualization.png', bbox_inches='tight'); plt.show()

def grad_stats(img):
    """độ lớn gradient trung bình và phân bố 9 bin hướng (vô hướng, 0-180°), cho từng kênh R,G,B"""
    out = {}
    for c, nm in enumerate('RGB'):
        ch = img[:, :, c].astype(np.float32)
        gx, gy = cv2.Sobel(ch, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(ch, cv2.CV_32F, 0, 1, ksize=3)
        mag = np.hypot(gx, gy); ang = (np.degrees(np.arctan2(gy, gx)) % 180)
        hist = np.bincount((ang // (180 / HOG_ORIENT)).astype(int).ravel().clip(0, HOG_ORIENT - 1),
                           weights=mag.ravel(), minlength=HOG_ORIENT)
        out[f'mag_{nm}'] = mag.mean()
        for b in range(HOG_ORIENT): out[f'bin{b}_{nm}'] = hist[b] / max(hist.sum(), 1e-9)
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    out['brightness'] = g.mean() / 255; out['contrast'] = g.std() / 255
    return out

an_m = pd.DataFrame([grad_stats(im) for im in IM_MAIN]).assign(label=df_main.label.values, dataset=df_main.dataset.values)
an_all = [an_m]
if len(df_ext):
    an_all.append(pd.DataFrame([grad_stats(im) for im in IM_EXT]).assign(label=df_ext.label.values, dataset=df_ext.dataset.values))
AN = pd.concat(an_all, ignore_index=True)
grad_tab = AN.groupby(['dataset', 'label']).mean(numeric_only=True).reindex(
    pd.MultiIndex.from_product([AN.dataset.unique(), LABELS]), fill_value=np.nan)
grad_tab.to_csv(OUT_DIR / 'gradient_stats_by_class.csv'); display(grad_tab[['mag_R', 'mag_G', 'mag_B', 'brightness', 'contrast']].round(3))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for c, nm in enumerate('RGB'):
    for l in LABELS:
        s = an_m[an_m.label == l]
        ax[c].plot(range(HOG_ORIENT), [s[f'bin{b}_{nm}'].mean() for b in range(HOG_ORIENT)], marker='o', label=l)
    ax[c].set_title(f'Phân bố hướng gradient – kênh {nm}'); ax[c].set_xlabel(f'bin hướng ({180 // HOG_ORIENT}°/bin)'); ax[c].legend(fontsize=8)
plt.tight_layout(); fig.savefig(OUT_DIR / 'orientation_bins_by_channel.png', bbox_inches='tight'); plt.show()

# --- không gian đặc trưng ---
sil = []
fig, axes = plt.subplots(len(HOG_VARIANTS), 2, figsize=(12, 4.6 * len(HOG_VARIANTS)), squeeze=False)
for r, v in enumerate(HOG_VARIANTS):
    key = (v, PPC_LIST[0]); Z = PCA(n_components=min(50, FEAT[key].shape[1]), random_state=SEED).fit(FEAT[key])
    Zt = Z.transform(FEAT[key]); ev = Z.explained_variance_ratio_
    for ci, (lab_arr, nm) in enumerate(((df_main.label.values, 'lớp (độ chín)'), (df_main.variety.values, 'giống chuối'))):
        for u in np.unique(lab_arr):
            m = lab_arr == u; axes[r, ci].scatter(Zt[m, 0], Zt[m, 1], s=6, alpha=.5, label=u)
        axes[r, ci].set_title(f'{v} (ppc={key[1]}) – PCA, tô theo {nm}', fontsize=9); axes[r, ci].legend(fontsize=7, markerscale=2)
    n_s = min(1200, len(Zt))
    sil.append(dict(variant=v, ppc=key[1], pca2_var=float(ev[:2].sum()), pca50_var=float(ev.sum()),
                    silhouette_by_class=float(silhouette_score(Zt, df_main.label.values, sample_size=n_s, random_state=SEED)),
                    silhouette_by_variety=float(silhouette_score(Zt, df_main.variety.values, sample_size=n_s, random_state=SEED))
                    if df_main.variety.nunique() > 1 else np.nan))
plt.tight_layout(); fig.savefig(OUT_DIR / 'feature_space_pca.png', bbox_inches='tight'); plt.show()
sil = pd.DataFrame(sil); sil.to_csv(OUT_DIR / 'feature_space_silhouette.csv', index=False); display(sil.round(3))
log('Silhouette (cao = tách tốt):', sil[['variant', 'silhouette_by_class', 'silhouette_by_variety']].round(3).to_dict('records'))

## 4. Thiết lập thí nghiệm (cross-validation theo quả)

| Bộ phân loại | Lưới tham số (chọn bằng CV lồng trong phần train) |
|---|---|
| KNN | k ∈ {1,3,5,7,9,15}, trọng số {đều, theo khoảng cách}, khoảng cách {euclid, cosine} |
| SVM tuyến tính | C ∈ {0.01, 0.1, 1, 10} |
| SVM RBF | C ∈ {1, 10, 100, 1000}, γ = (hệ số) / (số chiều × var(X)) với hệ số ∈ {0.03, 0.1, 0.3, 1, 3} – tức bội số của `gamma='scale'` tính **trên chính phần train** |

SVM dùng `class_weight='balanced'` (lệch lớp 1,51×). HOG đã được chuẩn hóa L2-Hys theo block nên **không** chuẩn hóa thêm từng chiều.

In [ ]:
class ScaledRBF(ClassifierMixin, BaseEstimator):
    """SVM RBF với gamma = gamma_mult / (n_features * var(X_train)) – bội số của gamma='scale', tính trong fit()."""
    def __init__(self, C=1.0, gamma_mult=1.0, class_weight='balanced'):
        self.C, self.gamma_mult, self.class_weight = C, gamma_mult, class_weight
    def fit(self, X, y):
        g = self.gamma_mult / (X.shape[1] * X.var() + 1e-12)
        self.m_ = SVC(kernel='rbf', C=self.C, gamma=g, class_weight=self.class_weight).fit(X, y)
        self.classes_ = self.m_.classes_; self.gamma_ = g; return self
    def predict(self, X): return self.m_.predict(X)

CLF_SPEC = {
    'KNN': (KNeighborsClassifier(algorithm='brute'),
            {'n_neighbors': [1, 3, 5, 7, 9, 15], 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'cosine']}),
    'SVM-linear': (SVC(kernel='linear', class_weight='balanced'), {'C': [0.01, 0.1, 1, 10]}),
    'SVM-RBF': (ScaledRBF(), {'C': [1, 10, 100, 1000], 'gamma_mult': [0.03, 0.1, 0.3, 1, 3]}),
}
def make_search(clf_name):
    est, grid = CLF_SPEC[clf_name]
    return GridSearchCV(clone(est), grid, cv=GroupKFold(N_INNER), scoring='f1_macro', n_jobs=N_JOBS, refit=True)

def within1(a, b): return float((np.abs(np.asarray(a) - np.asarray(b)) <= 1).mean())
def metrics(yt, yp):
    p, r, f, _ = precision_recall_fscore_support(yt, yp, labels=range(len(LABELS)), average='macro', zero_division=0)
    return dict(acc=accuracy_score(yt, yp), precision=p, recall=r, f1=f, within1=within1(yt, yp))

def run_cv(X, clf_name):
    cv = StratifiedGroupKFold(N_OUTER, shuffle=True, random_state=SEED)
    folds, oof, pred_ms = [], np.full(len(y), -1), []
    for k, (tr, te) in enumerate(cv.split(X, y, groups)):
        assert not set(groups[tr]) & set(groups[te]), 'Rò rỉ: một quả nằm ở cả train và test'
        t = time.time(); gs = make_search(clf_name).fit(X[tr], y[tr], groups=groups[tr]); fit_s = time.time() - t
        t = time.perf_counter(); pr = gs.predict(X[te]); pred_ms.append((time.perf_counter() - t) * 1000 / len(te))
        oof[te] = pr
        folds.append(dict(fold=k, n_train=len(tr), n_test=len(te), groups_train=len(set(groups[tr])), groups_test=len(set(groups[te])),
                          **{f'test_n_{LABELS[c]}': int((y[te] == c).sum()) for c in range(len(LABELS))},
                          **metrics(y[te], pr), inner_best_f1=gs.best_score_, best_params=json.dumps(gs.best_params_, default=str),
                          fit_seconds=fit_s))
    return pd.DataFrame(folds), oof, float(np.mean(pred_ms))

SUMMARY, FOLD_ROWS, OOF = [], [], {}
t_cv = time.time()
for (v, p) in KEYS:
    for cn in CLASSIFIERS:
        t = time.time(); folds, oof, pms = run_cv(FEAT[(v, p)], cn)
        for _, r in folds.iterrows(): FOLD_ROWS.append(dict(variant=v, ppc=p, classifier=cn, **r.to_dict()))
        row = dict(variant=v, ppc=p, classifier=cn, dims=FEAT[(v, p)].shape[1])
        for m in ('acc', 'precision', 'recall', 'f1', 'within1'):
            row[m + '_mean'], row[m + '_std'] = folds[m].mean(), folds[m].std()
        row['latency_ms_per_img'] = EXTRACT_MS[(v, p)] + pms; row['fit_seconds_mean'] = folds.fit_seconds.mean()
        SUMMARY.append(row); OOF[(v, p, cn)] = oof
        log(f'{v:<16} ppc={p:<3} {cn:<11} acc {row["acc_mean"]:.3f}±{row["acc_std"]:.3f} | macro-F1 {row["f1_mean"]:.3f}±{row["f1_std"]:.3f} | {time.time()-t:5.1f}s')
RUN_META['timings']['cv_s'] = round(time.time() - t_cv, 1)
res = pd.DataFrame(SUMMARY).sort_values('f1_mean', ascending=False).reset_index(drop=True)
res.to_csv(OUT_DIR / 'cv_results.csv', index=False)
pd.DataFrame(FOLD_ROWS).to_csv(OUT_DIR / 'cv_fold_metrics.csv', index=False)
log(f'Tổng thời gian CV: {RUN_META["timings"]["cv_s"] / 60:.1f} phút')

## 5. Kết quả cross-validation

In [ ]:
show = res.copy()
for m in ('acc', 'precision', 'recall', 'f1', 'within1'):
    show[m] = show[m + '_mean'].map('{:.3f}'.format) + ' ± ' + show[m + '_std'].map('{:.3f}'.format)
display(show[['variant', 'ppc', 'classifier', 'dims', 'acc', 'precision', 'recall', 'f1', 'within1', 'latency_ms_per_img']].round(2))
for p in PPC_LIST:
    print(f'\nMacro-F1 trung bình, ppc = {p}  (hàng: biến thể HOG, cột: bộ phân loại)')
    display(res[res.ppc == p].pivot(index='variant', columns='classifier', values='f1_mean').reindex(
        index=HOG_VARIANTS, columns=CLASSIFIERS).round(3))
fig, ax = plt.subplots(figsize=(10, 0.3 * len(res) + 1.5))
lab = res.variant + ' | ppc' + res.ppc.astype(str) + ' | ' + res.classifier
ax.barh(lab[::-1], res.f1_mean[::-1], xerr=res.f1_std[::-1], color='#66c2a5'); ax.set_xlabel('macro-F1 (mean ± std qua các fold)')
plt.tight_layout(); fig.savefig(OUT_DIR / 'cv_comparison.png', bbox_inches='tight'); plt.show()

### Cấu hình tốt nhất: confusion matrix, từng lớp, từng giống, từng góc chụp, ảnh bị nhầm
Dự đoán ngoài-fold: mỗi ảnh được dự đoán bởi mô hình chưa từng thấy quả chứa ảnh đó.

In [ ]:
b = res.iloc[0]; BEST = (b.variant, int(b.ppc), b.classifier); oofb = OOF[BEST]
log('Cấu hình tốt nhất theo CV:', BEST, f'| macro-F1 {b.f1_mean:.3f}±{b.f1_std:.3f}')
cm = confusion_matrix(y, oofb, labels=range(4)); cmn = cm / cm.sum(1, keepdims=True)
pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(OUT_DIR / 'cv_confusion_best.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for a, M, t, fmt in ((ax[0], cm, 'Số ảnh', 'd'), (ax[1], cmn, 'Tỉ lệ theo lớp thật', '.2f')):
    a.imshow(M, cmap='Blues'); a.grid(False); a.set_xticks(range(4)); a.set_yticks(range(4))
    a.set_xticklabels(LABELS, rotation=20); a.set_yticklabels(LABELS); a.set_xlabel('Dự đoán'); a.set_ylabel('Thực tế'); a.set_title(f'{t} – {BEST}', fontsize=9)
    for i in range(4):
        for j in range(4): a.text(j, i, format(M[i, j], fmt), ha='center', va='center', color='white' if M[i, j] > M.max() / 2 else 'black')
plt.tight_layout(); fig.savefig(OUT_DIR / 'cv_confusion_best.png', bbox_inches='tight'); plt.show()

pc, rc, fc, sc = precision_recall_fscore_support(y, oofb, labels=range(4), average=None, zero_division=0)
per_class = pd.DataFrame(dict(label=LABELS, precision=pc, recall=rc, f1=fc, support=sc)); per_class.to_csv(OUT_DIR / 'cv_per_class_best.csv', index=False); display(per_class.round(3))
df_main['pred_cv'] = [LABELS[i] for i in oofb]; df_main['correct'] = (oofb == y)
by_var = df_main.groupby('variety').correct.agg(['mean', 'size']).rename(columns={'mean': 'accuracy', 'size': 'n'})
by_view = df_main.groupby('view').correct.agg(['mean', 'size']).rename(columns={'mean': 'accuracy', 'size': 'n'})
by_vl = df_main.groupby(['variety', 'label']).correct.mean().unstack().reindex(columns=LABELS)
by_var.to_csv(OUT_DIR / 'cv_accuracy_by_variety.csv'); by_view.to_csv(OUT_DIR / 'cv_accuracy_by_view.csv'); by_vl.to_csv(OUT_DIR / 'cv_accuracy_by_variety_x_label.csv')
print('Accuracy theo giống:'); display(by_var.round(3)); print('Accuracy theo góc chụp:'); display(by_view.round(3))
print('Accuracy theo giống × lớp:'); display(by_vl.round(3))
mis = df_main[~df_main.correct][['file', 'path', 'group', 'variety', 'view', 'label', 'pred_cv']].rename(columns={'label': 'true'})
mis['adjacent'] = [abs(LABELS.index(a) - LABELS.index(b_)) == 1 for a, b_ in zip(mis.true, mis.pred_cv)]
mis.to_csv(OUT_DIR / 'cv_misclassified_best.csv', index=False)
pairs = pd.Series({(LABELS[i], LABELS[j]): int(cm[i, j]) for i in range(4) for j in range(4) if i != j}).sort_values(ascending=False)
log(f'Ảnh bị nhầm: {len(mis)}/{len(df_main)} ({len(mis)/len(df_main):.1%}); nhầm sang lớp kề: {mis.adjacent.mean():.1%}')
log('Các cặp nhầm nhiều nhất (thật -> đoán):', {f'{a}->{b_}': n for (a, b_), n in pairs.head(4).items()})
if len(mis):
    pick = mis.sample(min(8, len(mis)), random_state=SEED)
    fig, axes = plt.subplots(2, 4, figsize=(13, 6.4))
    for a, (i, r) in zip(axes.ravel(), pick.iterrows()):
        a.imshow(IM_MAIN[i]); a.axis('off'); a.set_title(f'thật: {r.true}\nđoán: {r.pred_cv}\n{r.file}', fontsize=7, color='crimson')
    for a in axes.ravel()[len(pick):]: a.axis('off')
    plt.tight_layout(); fig.savefig(OUT_DIR / 'cv_misclassified_examples.png', bbox_inches='tight'); plt.show()

### Độ nhạy tham số và đường cong học (cấu hình tốt nhất)
- Bảng `hyperparam_search_best.csv`: điểm CV trong của mọi tổ hợp tham số khi huấn luyện trên toàn bộ BananaID (cho biết kết quả nhạy với k / C / γ đến đâu).
- Đường cong học: macro-F1 trên fold test khi chỉ dùng một phần số quả của tập train (tham số cố định = tham số tốt nhất ở trên, nên chỉ dùng để xem hình dạng đường cong: còn tăng khi thêm dữ liệu hay đã bão hòa).

In [ ]:
Xb = FEAT[BEST[:2]]
final_cv = make_search(BEST[2]).fit(Xb, y, groups=groups)
hp = pd.DataFrame(final_cv.cv_results_)[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score')
hp.to_csv(OUT_DIR / 'hyperparam_search_best.csv', index=False); display(hp.head(8))
log('Tham số tốt nhất trên toàn bộ BananaID:', final_cv.best_params_, f'(macro-F1 CV trong {final_cv.best_score_:.3f}); '
    f'chênh lệch giữa tham số tốt nhất và tệ nhất: {hp.mean_test_score.max() - hp.mean_test_score.min():.3f}')

edge = [f'{k}={v}' for k, v in final_cv.best_params_.items()
        if isinstance(v, (int, float)) and len(CLF_SPEC[BEST[2]][1][k]) > 1 and v in (min(CLF_SPEC[BEST[2]][1][k]), max(CLF_SPEC[BEST[2]][1][k]))]
if edge:
    log('!! Tham số tốt nhất nằm ở BIÊN lưới tìm kiếm:', edge, '-> nên mở rộng lưới (kết quả có thể còn tốt hơn); ghi chú khi báo cáo.')

lc = []
cv = StratifiedGroupKFold(N_OUTER, shuffle=True, random_state=SEED)
for k, (tr, te) in enumerate(cv.split(Xb, y, groups)):
    ug = np.random.default_rng(SEED + k).permutation(np.unique(groups[tr]))
    for fr in LEARNING_CURVE_FRACS:
        idx = tr[np.isin(groups[tr], ug[:max(8, int(len(ug) * fr))])]
        est = clone(CLF_SPEC[BEST[2]][0]).set_params(**final_cv.best_params_).fit(Xb[idx], y[idx])
        lc.append(dict(fold=k, frac=fr, n_groups=len(set(groups[idx])), **metrics(y[te], est.predict(Xb[te]))))
lc = pd.DataFrame(lc); lcs = lc.groupby('frac').agg(n_groups=('n_groups', 'mean'), f1=('f1', 'mean'), f1_std=('f1', 'std'), acc=('acc', 'mean')).reset_index()
lcs.to_csv(OUT_DIR / 'learning_curve.csv', index=False); display(lcs.round(3))
fig, ax = plt.subplots(figsize=(6, 4)); ax.errorbar(lcs.n_groups, lcs.f1, yerr=lcs.f1_std, marker='o'); ax.set_xlabel('số quả dùng để huấn luyện'); ax.set_ylabel('macro-F1 (test fold)')
ax.set_title(f'Đường cong học – {BEST}', fontsize=9); plt.tight_layout(); fig.savefig(OUT_DIR / 'learning_curve.png', bbox_inches='tight'); plt.show()

## 6. Đánh giá cuối: tập test cùng miền + kiểm định ngoài

Với **cấu hình tốt nhất của từng biến thể HOG** (chọn bằng CV trên dev): huấn luyện lại trên toàn bộ dev (tham số chọn bằng `GroupKFold` theo quả) rồi đánh giá **một lần** trên
- **test BananaID** (quả chưa từng thấy, cùng thiết bị/nền) – kết quả "chuẩn" để báo cáo;
- **từng dataset ngoài** (BananaImageBD, BananaRipeness_GitHub). Ánh xạ nhãn giữa các bộ (Green→Unripe, Semi-ripe→Half_Ripe, Class A–D→Unripe…Overripe) **chỉ xấp xỉ** vì định nghĩa nhãn do các nhóm tác giả đặt riêng.

Kèm **chẩn đoán lệch miền**: độ giống (cosine) gần nhất của ảnh mỗi tập với dev, so với độ giống gần nhất giữa các quả khác nhau trong dev; nếu ảnh ngoài "xa" dev hơn hẳn thì kết quả tụt là do khác phân phối ảnh (thiết bị, nền, ánh sáng) chứ chưa chắc do mô hình kém.

In [ ]:
EVAL = {'test_' + TRAIN_DATASETS[0] if len(TRAIN_DATASETS) == 1 else 'test': (IM_TEST, y_test, df_test)}
for ds in (df_ext.dataset.unique() if len(df_ext) else []):
    mk = (df_ext.dataset == ds).values
    EVAL['ext_' + ds] = (IM_EXT[mk], y_ext[mk], df_ext[mk].reset_index(drop=True))
FINALS, EVAL_ROWS, PRED = {}, [], {}
best_per_variant = {v: res[res.variant == v].iloc[0] for v in HOG_VARIANTS}
for v, r in best_per_variant.items():
    key = (v, int(r.ppc), r.classifier)
    gs = make_search(key[2]).fit(FEAT[key[:2]], y, groups=groups); FINALS[key] = gs
    for en, (ims, ye, de) in EVAL.items():
        Xe, ems = hog_matrix(ims, key[0], key[1]); t = time.perf_counter(); pe = gs.predict(Xe); pms = (time.perf_counter() - t) * 1000 / len(Xe)
        PRED[(key, en)] = pe
        EVAL_ROWS.append(dict(eval_set=en, variant=v, ppc=key[1], classifier=key[2], params=json.dumps(gs.best_params_, default=str), cv_f1=r.f1_mean,
                              n=len(ye), **metrics(ye, pe), pred_dist=json.dumps(np.bincount(pe, minlength=4).tolist()),
                              true_dist=json.dumps(np.bincount(ye, minlength=4).tolist()), latency_ms_per_img=ems + pms))
eval_tab = pd.DataFrame(EVAL_ROWS); eval_tab['gap_f1'] = eval_tab.cv_f1 - eval_tab.f1
eval_tab.to_csv(OUT_DIR / 'final_eval_results.csv', index=False); display(eval_tab.drop(columns=['params', 'true_dist']).round(3))
for en in EVAL:
    log(f'[{en}]', eval_tab[eval_tab.eval_set == en][['variant', 'classifier', 'cv_f1', 'acc', 'f1', 'within1', 'pred_dist']].round(3).to_dict('records'))
for (key, en), pe in PRED.items():
    ye, de = EVAL[en][1], EVAL[en][2]
    pd.DataFrame(confusion_matrix(ye, pe, labels=range(4)), index=LABELS, columns=LABELS).to_csv(OUT_DIR / f'confusion_{en}_{key[0]}_ppc{key[1]}.csv')
    pd.DataFrame(dict(file=de.file.values, true=[LABELS[i] for i in ye], pred=[LABELS[i] for i in pe])).to_csv(
        OUT_DIR / f'predictions_{en}_{key[0]}_ppc{key[1]}.csv', index=False)

kb = BEST if BEST in FINALS else list(FINALS)[0]
fig, axes = plt.subplots(1, len(EVAL), figsize=(4.8 * len(EVAL), 4.4), squeeze=False)
for a, en in zip(axes[0], EVAL):
    ye = EVAL[en][1]; cme = confusion_matrix(ye, PRED[(kb, en)], labels=range(4))
    print(f'\nConfusion matrix – {en} – {kb}'); display(pd.DataFrame(cme, index=[f'thật: {l}' for l in LABELS], columns=[f'đoán: {l}' for l in LABELS]))
    print(classification_report(ye, PRED[(kb, en)], target_names=LABELS, digits=3, zero_division=0))
    a.imshow(cme / np.maximum(cme.sum(1, keepdims=True), 1), cmap='Oranges'); a.grid(False)
    a.set_xticks(range(4)); a.set_yticks(range(4)); a.set_xticklabels(LABELS, rotation=20, fontsize=7); a.set_yticklabels(LABELS, fontsize=7)
    for i in range(4):
        for j in range(4): a.text(j, i, cme[i, j], ha='center', va='center')
    a.set_title(f'{en} (số ảnh)\n{kb}', fontsize=8); a.set_xlabel('Dự đoán'); a.set_ylabel('Thực tế')
plt.tight_layout(); fig.savefig(OUT_DIR / 'final_confusion.png', bbox_inches='tight'); plt.show()

# --- chẩn đoán lệch miền, trên đặc trưng của cấu hình tốt nhất ---
Xi = FEAT[kb[:2]]
nz = lambda A: A / np.maximum(np.linalg.norm(A, axis=1, keepdims=True), 1e-9)
Si = nz(Xi) @ nz(Xi).T; Si[groups[:, None] == groups[None, :]] = -1          # bỏ cùng quả
dom_rows = [dict(nhom='dev -> dev (quả khác)', n=len(Xi), mean_nn_cosine=Si.max(1).mean(), median=np.median(Si.max(1)))]
XE = {}
for en, (ims, ye, de) in EVAL.items():
    XE[en], _ = hog_matrix(ims, kb[0], kb[1]); nn = (nz(XE[en]) @ nz(Xi).T).max(1)
    dom_rows.append(dict(nhom=f'{en} -> dev', n=len(nn), mean_nn_cosine=nn.mean(), median=np.median(nn)))
dom = pd.DataFrame(dom_rows); dom.to_csv(OUT_DIR / 'domain_shift_nn_similarity.csv', index=False); display(dom.round(3))
log('Lệch miền (độ giống cosine gần nhất tới dev):', dom.round(3).set_index('nhom').mean_nn_cosine.to_dict())
pca = PCA(2, random_state=SEED).fit(Xi); Zi = pca.transform(Xi)
fig, ax = plt.subplots(figsize=(6.4, 5)); ax.scatter(Zi[:, 0], Zi[:, 1], s=5, alpha=.3, label='dev')
for en in EVAL:
    Ze = pca.transform(XE[en]); ax.scatter(Ze[:, 0], Ze[:, 1], s=8, alpha=.6, label=en)
ax.set_title(f'PCA (fit trên dev) – {kb[0]}', fontsize=9); ax.legend(fontsize=7); plt.tight_layout(); fig.savefig(OUT_DIR / 'domain_shift_pca.png', bbox_inches='tight'); plt.show()
comp = AN.groupby('dataset')[['mag_R', 'mag_G', 'mag_B', 'brightness', 'contrast']].mean(); comp.to_csv(OUT_DIR / 'domain_shift_image_stats.csv'); display(comp.round(3))

# --- độ "thổi phồng" do rò rỉ: cùng cấu hình, huấn luyện trên train CŨ -> test CŨ của bước tiền xử lý ---
leak_note = ''
if use_regroup:
    dfa = pd.concat([df_main, df_test], ignore_index=True); Xa = np.vstack([FEAT[kb[:2]], XE[list(EVAL)[0]]]); ya = dfa.label.map(lab2i).values
    tr_o, te_o = (dfa.split == 'train').values, (dfa.split == 'test').values
    est = clone(CLF_SPEC[kb[2]][0]).set_params(**FINALS[kb].best_params_).fit(Xa[tr_o], ya[tr_o])
    leaky = metrics(ya[te_o], est.predict(Xa[te_o])); fair = eval_tab[(eval_tab.eval_set == list(EVAL)[0]) & (eval_tab.variant == kb[0])].iloc[0]
    pd.DataFrame([dict(split='cũ (rò rỉ theo quả)', **leaky), dict(split='chia lại theo quả', **{m_: fair[m_] for m_ in leaky})]).to_csv(OUT_DIR / 'leakage_inflation.csv', index=False)
    leak_note = f' (trên split cũ cùng cấu hình cho accuracy {leaky["acc"]:.3f} / macro-F1 {leaky["f1"]:.3f} so với {fair.acc:.3f} / {fair.f1:.3f} khi chia theo quả: chênh {leaky["f1"] - fair.f1:+.3f} macro-F1; rò rỉ làm tăng điểm nhưng với HOG ở đây ảnh hưởng nhỏ)'
    log('Thổi phồng do rò rỉ:', leak_note)

## 7. Mô hình cuối, demo và bản tóm tắt tự động

In [ ]:
PRIMARY = BEST
final = FINALS.get(PRIMARY) or make_search(PRIMARY[2]).fit(FEAT[PRIMARY[:2]], y, groups=groups)
joblib.dump(dict(model=final.best_estimator_, labels=LABELS, key=PRIMARY, img_size=IMG_SIZE, hog_orient=HOG_ORIENT), OUT_DIR / 'best_model.joblib')

def predict_image(path):
    """Dự đoán độ chín của một ảnh bất kỳ (cùng cách resize và HOG như lúc huấn luyện)."""
    return LABELS[int(final.predict(hog_feat(read_rgb(path), PRIMARY[0], PRIMARY[1])[None].astype(np.float32))[0])]

# demo: 4 ảnh đúng + 4 ảnh sai trên tập test cùng miền (tập chưa dùng để chọn gì)
en0 = list(EVAL)[0]; IMP, ytrue, ypred = EVAL[en0][0], EVAL[en0][1], PRED[(PRIMARY, en0)]
ok_i, bad_i = np.where(ypred == ytrue)[0], np.where(ypred != ytrue)[0]; rg = np.random.default_rng(SEED)
pick = list(rg.choice(ok_i, min(4, len(ok_i)), replace=False)) + list(rg.choice(bad_i, min(4, len(bad_i)), replace=False))
fig, axes = plt.subplots(2, 4, figsize=(13, 6.4))
for a, i in zip(axes.ravel(), pick):
    a.imshow(IMP[i]); a.axis('off'); a.set_title(f'thật: {LABELS[ytrue[i]]}\nđoán: {LABELS[ypred[i]]}', fontsize=9, color='green' if ytrue[i] == ypred[i] else 'crimson')
for a in axes.ravel()[len(pick):]: a.axis('off')
fig.suptitle(f'Demo trên {en0} – xanh: đúng, đỏ: sai'); plt.tight_layout(); fig.savefig(OUT_DIR / 'demo_examples.png', bbox_inches='tight'); plt.show()

# --- bản tóm tắt tự động ---
def ev(en, key=PRIMARY): return eval_tab[(eval_tab.eval_set == en) & (eval_tab.variant == key[0])].iloc[0]
L = ['# Tóm tắt thực nghiệm HOG (tự sinh)', '',
     f'- Dữ liệu: dev {len(df_main):,} ảnh / {df_main.banana.nunique()} quả {cnt(df_main)}; test {len(df_test):,} ảnh / {df_test.banana.nunique()} quả {cnt(df_test)}; '
     f'ngoài {len(df_ext):,} ảnh ({df_ext.dataset.value_counts().to_dict() if len(df_ext) else {}})',
     f'- Split: {"CHIA LẠI theo quả" if use_regroup else "dùng split có sẵn"}; split có sẵn của bước tiền xử lý có {n_leak} quả nằm ở >1 split '
     f'({leak_tab.set_index("split").frac["test"]:.1%} ảnh test có cùng quả trong train/val)' + leak_note,
     f'- Cấu hình tốt nhất theo CV (dev): **{BEST}** – macro-F1 {b.f1_mean:.3f} ± {b.f1_std:.3f}, accuracy {b.acc_mean:.3f} ± {b.acc_std:.3f}, '
     f'macro-precision {b.precision_mean:.3f}, macro-recall {b.recall_mean:.3f}',
     f'- Tập test cùng miền ({en0}, huấn luyện lại trên toàn dev): accuracy {ev(en0).acc:.3f}, macro-precision {ev(en0).precision:.3f}, '
     f'macro-recall {ev(en0).recall:.3f}, macro-F1 {ev(en0).f1:.3f}, lệch ≤1 mức {ev(en0).within1:.3f}',
     f'- Tham số chọn (toàn dev): {final_cv.best_params_}; độ nhạy tham số (max-min macro-F1 CV trong): {hp.mean_test_score.max() - hp.mean_test_score.min():.3f}',
     '- So sánh biến thể HOG (macro-F1 CV tốt nhất của mỗi biến thể): ' + ', '.join(f'{v} {res[res.variant == v].f1_mean.max():.3f}' for v in HOG_VARIANTS),
     '- So sánh bộ phân loại (tốt nhất của mỗi loại): ' + ', '.join(f'{c} {res[res.classifier == c].f1_mean.max():.3f}' for c in CLASSIFIERS),
     (f'- Lệch ≤ 1 mức (CV): {b.within1_mean:.3f}; ảnh nhầm {len(mis)}/{len(df_main)}, trong đó nhầm sang lớp kề {mis.adjacent.mean():.1%}' if len(mis) else '- Không có ảnh nhầm'),
     '- Cặp nhầm nhiều nhất: ' + ', '.join(f'{a}→{b_} ({n})' for (a, b_), n in pairs.head(3).items()),
     f'- Accuracy CV theo giống: {by_var.accuracy.round(3).to_dict()}; theo góc chụp: {by_view.accuracy.round(3).to_dict()}',
     f'- Đường cong học: macro-F1 {lcs.f1.iloc[0]:.3f} (n={lcs.n_groups.iloc[0]:.0f} quả) → {lcs.f1.iloc[-1]:.3f} (n={lcs.n_groups.iloc[-1]:.0f} quả); '
     f'tăng {lcs.f1.iloc[-1] - lcs.f1.iloc[-2]:+.3f} ở bước cuối ({"còn tăng – thêm dữ liệu có thể giúp" if lcs.f1.iloc[-1] - lcs.f1.iloc[-2] > 0.01 else "gần bão hòa"})',
     f"- Silhouette: theo lớp {sil.set_index('variant').silhouette_by_class.round(3).to_dict()} so với theo giống {sil.set_index('variant').silhouette_by_variety.round(3).to_dict()} "
     '(nếu theo giống cao hơn hẳn: đặc trưng HOG phản ánh hình dạng/giống nhiều hơn độ chín)']
for en in EVAL:
    if en == en0: continue
    e = ev(en); gap = b.f1_mean - e.f1
    L += [f'- Kiểm định ngoài {en} (cùng cấu hình {PRIMARY[0]}/{PRIMARY[2]}): accuracy {e.acc:.3f}, macro-F1 {e.f1:.3f} so với CV {b.f1_mean:.3f} (chênh {gap:+.3f}); '
          f'phân bố dự đoán {e.pred_dist} so với thật {e.true_dist}']
    if gap > 0.2: L.append(f'  - **Cảnh báo:** {en} thấp hơn CV rất nhiều → lệch miền (thiết bị/nền/ánh sáng/định nghĩa nhãn) hoặc mô hình dựa vào đặc điểm riêng của {TRAIN_DATASETS}; nêu trong phần hạn chế.')
L += ['- Lệch miền (độ giống cosine gần nhất tới dev): ' + str(dom.round(3).set_index('nhom').mean_nn_cosine.to_dict()),
      '', 'Lưu ý: ánh xạ nhãn giữa các dataset chỉ xấp xỉ.']
(OUT_DIR / 'summary.md').write_text('\n'.join(L), encoding='utf-8'); print('\n'.join(L))
RUN_META['finished'] = time.strftime('%Y-%m-%d %H:%M:%S'); RUN_META['best_cv'] = dict(key=list(BEST), f1=float(b.f1_mean))
json.dump(RUN_META, open(OUT_DIR / 'run_meta.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=str)
print('\nFile đã lưu trong', OUT_DIR.resolve()); [print(f'  {f.stat().st_size / 1024:>9.1f} KB  {f.name}') for f in sorted(OUT_DIR.iterdir())]